# 19A4E — Post-test Cycle-25 Score-Shift Diagnostic

## Purpose

Diagnose the strong **2025 operating-threshold collapse** observed after the
primary frozen 2021–2025 evaluation, without changing any scientific decision.

This is a **post-test diagnostic only**.

### Frozen quantities that MUST NOT change
- CNN-GRU weights
- AIA normalisation
- Platt coefficient/intercept
- operating threshold
- primary 2021–2025 scorecard

### Questions
1. Did the raw-score distribution shift in 2025?
2. Did the calibrated-probability distribution shift in 2025?
3. Is ranking still preserved within 2025 despite the operating-point failure?
4. How did positive and negative score distributions move by year?
5. What fraction of positives/negatives lies above the frozen threshold?
6. Is 2025 qualitatively different from 2021–2024 at the score-distribution level?

No new threshold is selected. Any hypothetical threshold shown is forbidden.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, average_precision_score

HOME = Path.home()
PRED = HOME / "aia19_cycle25_final_evaluation" / "cycle25_2021_2025_frozen_predictions.csv.gz"
OUT = HOME / "aia19_cycle25_shift_diagnostic"
OUT.mkdir(parents=True, exist_ok=True)

FROZEN_THRESHOLD = 0.030438695842933242

assert PRED.exists(), PRED
df = pd.read_csv(PRED)

assert len(df) == 49329
assert df["target_sample_id"].nunique() == 49329
assert int(df["y_true"].sum()) == 2351

print("Rows:", len(df))
print("Positives:", int(df["y_true"].sum()))
print("Years:", sorted(df["stored_year"].unique().tolist()))


## 1. Year-wise score-distribution summary

In [ ]:
def q(x, p):
    return float(np.quantile(np.asarray(x, dtype=float), p))

rows = []

for year, g in df.groupby("stored_year"):
    for cls_name, gg in [
        ("all", g),
        ("negative", g[g["y_true"].eq(0)]),
        ("positive", g[g["y_true"].eq(1)]),
    ]:
        rows.append({
            "year": int(year),
            "class": cls_name,
            "n": int(len(gg)),
            "raw_logit_mean": float(gg["raw_logit"].mean()),
            "raw_logit_std": float(gg["raw_logit"].std()),
            "raw_logit_q01": q(gg["raw_logit"], .01),
            "raw_logit_q25": q(gg["raw_logit"], .25),
            "raw_logit_q50": q(gg["raw_logit"], .50),
            "raw_logit_q75": q(gg["raw_logit"], .75),
            "raw_logit_q99": q(gg["raw_logit"], .99),
            "cal_prob_mean": float(gg["calibrated_probability"].mean()),
            "cal_prob_q01": q(gg["calibrated_probability"], .01),
            "cal_prob_q25": q(gg["calibrated_probability"], .25),
            "cal_prob_q50": q(gg["calibrated_probability"], .50),
            "cal_prob_q75": q(gg["calibrated_probability"], .75),
            "cal_prob_q99": q(gg["calibrated_probability"], .99),
            "fraction_above_frozen_threshold": float(
                (gg["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
            ),
        })

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "year_class_score_distribution_summary.csv", index=False)

print(summary.to_string(index=False))


## 2. Operating-point decomposition by year

In [ ]:
op_rows = []

for year, g in df.groupby("stored_year"):
    pos = g[g["y_true"].eq(1)]
    neg = g[g["y_true"].eq(0)]

    op_rows.append({
        "year": int(year),
        "n": int(len(g)),
        "positives": int(len(pos)),
        "prevalence": float(g["y_true"].mean()),
        "positive_fraction_above_threshold": float(
            (pos["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
        ),
        "negative_fraction_above_threshold": float(
            (neg["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
        ),
        "positive_median_calibrated_probability": float(
            pos["calibrated_probability"].median()
        ),
        "negative_median_calibrated_probability": float(
            neg["calibrated_probability"].median()
        ),
        "positive_median_raw_logit": float(pos["raw_logit"].median()),
        "negative_median_raw_logit": float(neg["raw_logit"].median()),
        "roc_auc": float(roc_auc_score(g["y_true"], g["calibrated_probability"])),
        "pr_auc": float(average_precision_score(g["y_true"], g["calibrated_probability"])),
    })

op = pd.DataFrame(op_rows)
op.to_csv(OUT / "yearwise_operating_point_decomposition.csv", index=False)

print(op.to_string(index=False))


## 3. Distribution-shift tests against 2024

These tests are descriptive diagnostics only.

We compare 2025 with 2024 using two-sample Kolmogorov–Smirnov statistics for:
- raw logits;
- calibrated probabilities;
- positives only;
- negatives only.

A small p-value indicates that the score distributions differ, but does not by itself identify the physical cause.


In [ ]:
g24 = df[df["stored_year"].eq(2024)]
g25 = df[df["stored_year"].eq(2025)]

tests = []

for col in ["raw_logit", "calibrated_probability"]:
    for cls_name, a, b in [
        ("all", g24, g25),
        ("positive", g24[g24["y_true"].eq(1)], g25[g25["y_true"].eq(1)]),
        ("negative", g24[g24["y_true"].eq(0)], g25[g25["y_true"].eq(0)]),
    ]:
        stat, pvalue = ks_2samp(a[col].to_numpy(), b[col].to_numpy())
        tests.append({
            "reference_year": 2024,
            "comparison_year": 2025,
            "class": cls_name,
            "variable": col,
            "ks_statistic": float(stat),
            "p_value": float(pvalue),
            "n_2024": int(len(a)),
            "n_2025": int(len(b)),
        })

ks = pd.DataFrame(tests)
ks.to_csv(OUT / "ks_2024_vs_2025_score_shift.csv", index=False)
print(ks.to_string(index=False))


## 4. Score distributions by year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for year in sorted(df["stored_year"].unique()):
    vals = df.loc[df["stored_year"].eq(year), "calibrated_probability"].to_numpy()
    ax.hist(
        vals,
        bins=120,
        histtype="step",
        density=True,
        label=str(year),
    )

ax.axvline(FROZEN_THRESHOLD, linestyle="--", linewidth=1.5, label="Frozen threshold")
ax.set_xlim(0, np.quantile(df["calibrated_probability"], 0.995))
ax.set_xlabel("Calibrated probability")
ax.set_ylabel("Density")
ax.set_title("Frozen Cycle-25 calibrated-probability distributions")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "calibrated_probability_distribution_by_year.png", dpi=180)
plt.close(fig)

print("Saved calibrated_probability_distribution_by_year.png")


## 5. Positive-class distributions by year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

pos_df = df[df["y_true"].eq(1)]

for year in sorted(pos_df["stored_year"].unique()):
    vals = pos_df.loc[pos_df["stored_year"].eq(year), "calibrated_probability"].to_numpy()
    ax.hist(
        vals,
        bins=100,
        histtype="step",
        density=True,
        label=str(year),
    )

ax.axvline(FROZEN_THRESHOLD, linestyle="--", linewidth=1.5, label="Frozen threshold")
ax.set_xlim(0, np.quantile(pos_df["calibrated_probability"], 0.995))
ax.set_xlabel("Calibrated probability")
ax.set_ylabel("Density")
ax.set_title("Positive-class calibrated-probability distributions")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "positive_calibrated_probability_distribution_by_year.png", dpi=180)
plt.close(fig)

print("Saved positive_calibrated_probability_distribution_by_year.png")


## 6. Yearly probability quantiles

In [ ]:
quantile_rows = []

for year, g in df.groupby("stored_year"):
    for label in [0, 1]:
        gg = g[g["y_true"].eq(label)]
        rec = {
            "year": int(year),
            "y_true": int(label),
            "n": int(len(gg)),
        }
        for p in [0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99]:
            rec[f"q{int(p*100):02d}"] = float(
                np.quantile(gg["calibrated_probability"], p)
            )
        quantile_rows.append(rec)

quantiles = pd.DataFrame(quantile_rows)
quantiles.to_csv(OUT / "year_class_calibrated_probability_quantiles.csv", index=False)
print(quantiles.to_string(index=False))


## 7. Diagnostic interpretation record

In [ ]:
op_idx = op.set_index("year")

diagnostic = {
    "status": "POST_TEST_SCORE_SHIFT_DIAGNOSTIC_COMPLETE_NO_TUNING",
    "primary_test_result_unchanged": True,
    "frozen_threshold": FROZEN_THRESHOLD,
    "year_2024": op_idx.loc[2024].to_dict(),
    "year_2025": op_idx.loc[2025].to_dict(),
    "ks_2024_vs_2025": ks.to_dict(orient="records"),
    "model_weights_updated": False,
    "calibrator_refit": False,
    "threshold_reselected": False,
    "metrics_reoptimised": False,
    "notes": [
        "This notebook is diagnostic only and does not alter the frozen 19A4 result.",
        "ROC-AUC/PR-AUC characterize ranking; the frozen operating point characterizes threshold transfer.",
        "A large shift in positive score distribution with preserved ranking is consistent with score/probability distribution shift.",
        "Statistical distribution shift alone does not establish whether the cause is solar-regime shift, acquisition/preprocessing shift, or both.",
        "AIA source-image statistics and production metadata should be compared separately before attributing cause."
    ],
}

(OUT / "diagnostic_record.json").write_text(json.dumps(diagnostic, indent=2) + "\n")

print(json.dumps(diagnostic, indent=2))
print("\n19A4E_SCORE_SHIFT_DIAGNOSTIC_COMPLETE")
